In [21]:
import re, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

In [22]:
# Only silence the noisy pandas/sklearn deprecations you have actually
warnings.filterwarnings("ignore", category=FutureWarning)

In [23]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
 
CYCLES = ["E", "F", "G"]
FILES = ["DEMO", "BMX", "CBC", "ENX", "MCQ", "OCQ", "RDQ", "SMQ", "SPX"]
WT_COL, PSU_COL, STRATA_COL = "WTMEC2YR", "SDMVPSU", "SDMVSTRA"
N_CYCLES = len(CYCLES)

In [13]:
# Resolve the data root relative to wherever the kernel started, so the
# notebook works whether it is launched from Code/ or from the repo root.
_CANDIDATES = [Path("../data/raw"), Path("data/raw"), Path("../../data/raw")]
SEARCH_ROOT = next((p.resolve() for p in _CANDIDATES if p.is_dir()), None)
if SEARCH_ROOT is None:
    raise FileNotFoundError(
        "No data/raw directory found. Looked in: "
        + ", ".join(str(p.resolve()) for p in _CANDIDATES)
    )
 
OUT_DIR = Path(SEARCH_ROOT).parent / "proc"
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
print("Search root:", SEARCH_ROOT)
print("Output dir :", OUT_DIR)

Search root: /Users/farhan/projects/Asthma-COPD-V2/data/raw
Output dir : /Users/farhan/projects/Asthma-COPD-V2/data/proc


In [24]:
def discover_nhanes_files(search_root=None):
    """Map (module, cycle) -> path for every NHANES .xpt under search_root."""
    root = Path(search_root) if search_root is not None else SEARCH_ROOT
    file_map, duplicates = {}, {}
    pattern = re.compile(r"(DEMO|BMX|CBC|ENX|MCQ|OCQ|RDQ|SMQ|SPX)_([EFG])")

    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() != ".xpt":
            continue
        stem = re.sub(r"\(\d+\)$", "", path.stem.upper().replace(" ", ""))
        m = pattern.fullmatch(stem)
        if not m:
            continue
        key = m.groups()
        if key in file_map:
            duplicates.setdefault(key, [file_map[key]]).append(path)
        # Prefer a filename with no "(n)" duplicate marker.
        if key not in file_map or "(" not in path.stem:
            file_map[key] = path

    for key, paths in duplicates.items():
        print(f"  ! {key[0]}_{key[1]}: {len(paths)} candidates, using {file_map[key].name}")

    return file_map


XPT_FILE_MAP = discover_nhanes_files()
expected = {(mod, cyc) for cyc in CYCLES for mod in FILES}
missing = sorted(expected - set(XPT_FILE_MAP))

print(f"Detected {len(XPT_FILE_MAP)} of {len(expected)} required XPT files.")
if missing:
    raise FileNotFoundError("Missing: " + ", ".join(f"{m}_{c}" for m, c in missing))

Detected 27 of 27 required XPT files.


In [25]:
def load_xpt(module, cycle):
    path = XPT_FILE_MAP[(module, cycle)]
    df = pd.read_sas(path, format="xport", encoding="latin-1")
    df.columns = [str(c).strip().upper() for c in df.columns]
    if "SEQN" not in df.columns:
        raise KeyError(f"SEQN missing from {path.name}")
    if df["SEQN"].duplicated().any():
        raise ValueError(f"{path.name} has duplicate SEQN")
    return df

def load_cycle(cycle):
    merged = load_xpt("DEMO", cycle).copy()
    for module in FILES[1:]:
        right = load_xpt(module, cycle).copy()
        overlap = [c for c in right.columns if c != "SEQN" and c in merged.columns]
        if overlap:
            right = right.rename(columns={c: f"{c}_{module}" for c in overlap})
        merged = merged.merge(right, on="SEQN", how="left", validate="one_to_one")
    merged["CYCLE"] = cycle
    return merged

def build_merged(cycles=CYCLES):
    return pd.concat([load_cycle(c) for c in cycles],
                     axis=0, ignore_index=True, sort=False)

full = build_merged()

print("Merged shape:", full.shape)
print("Rows per cycle:", full["CYCLE"].value_counts().sort_index().to_dict())
for c in (WT_COL, PSU_COL, STRATA_COL):
    print(f"  {c}: present={c in full.columns}, missing={full[c].isna().sum()}")

Merged shape: (30442, 357)
Rows per cycle: {'E': 10149, 'F': 10537, 'G': 9756}
  WTMEC2YR: present=True, missing=0
  SDMVPSU: present=True, missing=0
  SDMVSTRA: present=True, missing=0


In [26]:
REQUIRED_COLS = [
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "INDFMPIR",
    WT_COL, PSU_COL, STRATA_COL,
    "MCQ010", "MCQ035", "MCQ160G", "MCQ160K",
    "SPXNFEV1", "SPXNFVC", "SPXNPEF", "SPXBFEV1", "SPXBFVC",
    "BMXBMI", "BMXWAIST",
    "LBXWBCSI", "LBXEOPCT", "LBDEONO", "ENXMEAN",
    "SMQ020", "SMQ040", "OCD150",
    "RDQ031", "RDQ070", "RDQ080", "RDQ090", "RDQ100", "RDQ140",
]

absent = [c for c in REQUIRED_COLS if c not in full.columns]
collided = {
    c: [f"{c}_{m}" for m in FILES if f"{c}_{m}" in full.columns]
    for c in REQUIRED_COLS
}
collided = {k: v for k, v in collided.items() if v}

if collided:
    raise RuntimeError(f"Required columns were renamed by merge collision: {collided}")
if absent:
    raise KeyError(f"Required columns absent after merge: {absent}")

print(f"All {len(REQUIRED_COLS)} required columns present under expected names.")
print("Rows per cycle:", full["CYCLE"].value_counts().sort_index().to_dict())

All 33 required columns present under expected names.
Rows per cycle: {'E': 10149, 'F': 10537, 'G': 9756}


In [27]:
def phenotype_flags(full):
    """Phenotype indicators shared by the analytic cohort and the survey frame."""
    age_ok = full["RIDAGEYR"].between(40, 79)
    asthma = (full["MCQ010"] == 1) & (full["MCQ035"] == 1)

    post_ok = full["SPXBFEV1"].notna() & full["SPXBFVC"].notna()
    pre_ok = full["SPXNFEV1"].notna() & full["SPXNFVC"].notna()

    fev1 = np.where(post_ok, full["SPXBFEV1"],
                    np.where(pre_ok, full["SPXNFEV1"], np.nan))
    fvc = np.where(post_ok, full["SPXBFVC"],
                   np.where(pre_ok, full["SPXNFVC"], np.nan))
    fev1 = pd.Series(fev1, index=full.index)
    fvc = pd.Series(fvc, index=full.index)
    ratio = fev1 / fvc

    copd_self = (full["MCQ160G"] == 1) | (full["MCQ160K"] == 1)
    copd = copd_self & (ratio < 0.70)

    return pd.DataFrame({
        "age_ok": age_ok,
        "asthma": asthma,
        "copd": copd,
        "copd_self": copd_self,
        "fev1": fev1,
        "fvc": fvc,
        "ratio": ratio,
        "label_source": np.where(post_ok, "post_bd",
                                 np.where(pre_ok, "pre_bd", "none")),
    }, index=full.index)


def build_cohort(full):
    f = phenotype_flags(full)
    overlap = f["asthma"] & f["copd"]
    keep = f["age_ok"] & (f["asthma"] | f["copd"]) & (~overlap)

    cohort = full.loc[keep].copy()
    cohort["target"] = f.loc[keep, "asthma"].astype(int).to_numpy()
    cohort["outcome_fev1"] = f.loc[keep, "fev1"].to_numpy()
    cohort["outcome_fvc"] = f.loc[keep, "fvc"].to_numpy()
    cohort["outcome_fev1_fvc_ratio"] = f.loc[keep, "ratio"].to_numpy()
    cohort["label_source"] = f.loc[keep, "label_source"].to_numpy()
    return cohort


flags = phenotype_flags(full)
cohort = build_cohort(full)

# Plain dict, not cohort.attrs — attrs does not survive a parquet round-trip.
COHORT_COUNTS = {
    "n_age_eligible_either": int((flags["age_ok"] & (flags["asthma"] | flags["copd"])).sum()),
    "n_overlap_excluded": int((flags["age_ok"] & flags["asthma"] & flags["copd"]).sum()),
    "n_cohort": int(len(cohort)),
    "n_asthma": int((cohort["target"] == 1).sum()),
    "n_copd": int((cohort["target"] == 0).sum()),
}

print("Cohort:", cohort.shape)
for k, v in COHORT_COUNTS.items():
    print(f"  {k:26s} {v}")

Cohort: (922, 362)
  n_age_eligible_either      995
  n_overlap_excluded         73
  n_cohort                   922
  n_asthma                   794
  n_copd                     128


In [28]:
copd_rows = cohort.loc[cohort["target"] == 0]
src = copd_rows["label_source"].value_counts()

print("COPD-only labelling provenance:")
for k in ("post_bd", "pre_bd"):
    n = int(src.get(k, 0))
    print(f"  {k:8s} {n:4d}  ({n / len(copd_rows) * 100:5.1f}% of {len(copd_rows)})")

mixed = (
    (full["SPXBFEV1"].notna() ^ full["SPXBFVC"].notna())
    & full["SPXNFEV1"].notna()
    & full["SPXNFVC"].notna()
)
print(f"\nMixed-manoeuvre rows avoided by requiring both components: {int(mixed.sum())}")

COPD-only labelling provenance:
  post_bd    38  ( 29.7% of 128)
  pre_bd     90  ( 70.3% of 128)

Mixed-manoeuvre rows avoided by requiring both components: 0


In [29]:
def build_survey_frame(full):
    f = phenotype_flags(full)
    svy = full.copy()

    svy["age_eligible"] = f["age_ok"].astype(bool)
    svy["asthma_only"] = (f["asthma"] & ~f["copd"]).astype(float)
    svy["copd_only"] = (f["copd"] & ~f["asthma"]).astype(float)
    svy["overlap"] = (f["asthma"] & f["copd"]).astype(float)
    svy["no_disease"] = (~f["asthma"] & ~f["copd"]).astype(float)

    # 1.0 = asthma-only, 0.0 = COPD-only, NaN = outside the analytic cohort.
    svy["group"] = np.select(
        [f["age_ok"] & (svy["asthma_only"] == 1),
         f["age_ok"] & (svy["copd_only"] == 1)],
        [1.0, 0.0],
        default=np.nan,
    )

    svy["w6yr"] = svy[WT_COL] / N_CYCLES
    return svy


svy = build_survey_frame(full)

# The four categories must partition the age-eligible population, otherwise the
# prevalence estimates will not sum to 100%.
_part = svy.loc[svy["age_eligible"],
                ["asthma_only", "copd_only", "overlap", "no_disease"]]
assert (_part.sum(axis=1) == 1).all(), "Phenotype categories do not partition the sample"

# The survey frame and the analytic cohort must agree on group membership.
assert int((svy["group"] == 1).sum()) == COHORT_COUNTS["n_asthma"]
assert int((svy["group"] == 0).sum()) == COHORT_COUNTS["n_copd"]

print(f"Survey frame: {svy.shape[0]} rows, {int(svy['age_eligible'].sum())} age-eligible")
print("Group counts:", svy["group"].value_counts(dropna=True).to_dict())
print("Strata:", svy[STRATA_COL].nunique(),
      "| PSUs:", svy.groupby(STRATA_COL)[PSU_COL].nunique().sum())
print("Zero/missing MEC weight among age-eligible:",
      int((svy.loc[svy["age_eligible"], WT_COL].fillna(0) <= 0).sum()))

Survey frame: 30442 rows, 10536 age-eligible
Group counts: {1.0: 794, 0.0: 128}
Strata: 45 | PSUs: 94
Zero/missing MEC weight among age-eligible: 0


In [34]:
cohort.to_parquet(OUT_DIR / "cohort.parquet", index=True)
svy.to_parquet(OUT_DIR / "survey_frame.parquet", index=True)

with open(OUT_DIR / "cohort_counts.json", "w") as fh:
    json.dump(COHORT_COUNTS, fh, indent=2)

print("Wrote:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:26s} {p.stat().st_size / 1e6:7.1f} MB")

Wrote:
  cohort.csv                     0.9 MB
  cohort.parquet                 0.4 MB
  cohort_counts.json             0.0 MB
  survey_frame.parquet           4.4 MB
